<a href="https://colab.research.google.com/github/Renisha-bit/Zomathon/blob/main/implementation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import torch.nn as nn

class CSAORailModel(nn.Module):
    def __init__(self, vocab_size, embed_dim):
        super().__init__()
        # 1. Embeddings for Items and Context
        self.item_embeddings = nn.Embedding(vocab_size, embed_dim)
        self.context_embeddings = nn.Linear(10, embed_dim)

        # 2. Transformer to understand Cart Sequence
        self.cart_transformer = nn.TransformerEncoderLayer(d_model=embed_dim, nhead=4)

        # 3. Final Ranking Head
        self.fc = nn.Sequential(
            nn.Linear(embed_dim * 2, 128),
            nn.ReLU(),
            nn.Linear(128, 1),
            nn.Sigmoid() # Predicts probability of 'Add-to-Cart'
        )

    def forward(self, cart_items, user_context, candidate_item):
        # Process Cart through Transformer
        cart_repr = self.item_embeddings(cart_items)
        cart_repr = self.cart_transformer(cart_repr).mean(dim=1)

        # Process Candidate
        cand_repr = self.item_embeddings(candidate_item)

        # Combine and Score
        combined = torch.cat([cart_repr, cand_repr], dim=-1)
        return self.fc(combined)